# IMDb Catalogue Analytics

**Grace Ngo | SQL and Python**

How do eligibility rules and vote floors alter catalogue summaries and ranked shortlists? This walkthrough runs the reviewed package on fictional records. Original IMDb results cannot be reproduced without the missing source database.


In [1]:
from pathlib import Path
import sys
import json
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'imdb_analytics').is_dir()), None)
if root is None:
    raise RuntimeError('Open this notebook from the repository directory.')
sys.path.insert(0, str(root))
from imdb_analytics.fixture import demo_connection
from imdb_analytics.analysis import run, profile_database, validate_database, vote_sensitivity
connection = demo_connection()
print('FICTIONAL VERIFICATION DATA. Original imdb.db was not supplied.')


FICTIONAL VERIFICATION DATA. Original imdb.db was not supplied.


## SQL aggregation and eligibility

Runtime bins contain explicit mean title ratings and support counts. Genre membership uses exact tokens; multi-genre shares use all eligible movies as the denominator. Adult titles and television entries are excluded consistently.


In [2]:
report = run(connection, root / 'reports/demo', 'synthetic_fixture')
print(json.dumps(report, indent=2))


{
  "author": "Grace Ngo",
  "dataset_kind": "synthetic_fixture",
  "year": 2010,
  "profile": {
    "titles": 41,
    "ratings": 41,
    "non_adult_movies": 39,
    "missing_runtime_movies": 1,
    "missing_genre_movies": 1,
    "movies_without_rating": 0,
    "orphan_rating_rows": 0
  },
  "parameters": {
    "genres": [
      "Drama",
      "Comedy",
      "Action",
      "Documentary"
    ],
    "runtime_interval": "[40,200)",
    "bin_width": 10,
    "min_votes_runtime": 200,
    "min_titles_per_bin": 2,
    "coverage_years": [
      1920,
      2019
    ],
    "ranking_min_votes": 5000,
    "top_n": 25,
    "vote_thresholds": [
      200,
      5000,
      50000
    ]
  },
  "vote_sensitivity": [
    {
      "year": 2010,
      "minimum_votes": 200,
      "eligible_titles": 18,
      "returned_titles": 18,
      "retained_from_baseline": 18,
      "jaccard_with_baseline": 1.0,
      "baseline_retention_share": 1.0
    },
    {
      "year": 2010,
      "minimum_votes": 5000,
    

## Vote-floor sensitivity

These examples test code behavior, not IMDb audience patterns. Higher floors shrink the eligible pool. Shortlist overlap documents the effect of that choice without treating votes as views or as proof of statistical reliability.


In [3]:
for row in vote_sensitivity(connection, year=2010, thresholds=[200, 5000, 50000], top_n=25):
    print(row)


{'year': 2010, 'minimum_votes': 200, 'eligible_titles': 18, 'returned_titles': 18, 'retained_from_baseline': 18, 'jaccard_with_baseline': 1.0, 'baseline_retention_share': 1.0}
{'year': 2010, 'minimum_votes': 5000, 'eligible_titles': 18, 'returned_titles': 18, 'retained_from_baseline': 18, 'jaccard_with_baseline': 1.0, 'baseline_retention_share': 1.0}
{'year': 2010, 'minimum_votes': 50000, 'eligible_titles': 12, 'returned_titles': 12, 'retained_from_baseline': 12, 'jaccard_with_baseline': 0.6666666666666666, 'baseline_retention_share': 0.6666666666666666}


## Use authorized local data

Run `python -m imdb_analytics.analysis --database data/imdb.db --output outputs/local` from the repository root. A streaming importer also supports locally supplied official TSV files. Real source files and generated local results are excluded from Git. See the README for schema, provenance and interpretation limits.
